# Linguistic Style Analysis Across Classic Literature

Ingrid Foslien | September 2026

This notebook explores differences in vocabulary richness, sentence
structure, and word usage across classic novels using NLTK's Gutenberg
corpus. It's an early step toward a full interactive dashboard comparing
linguistic style across authors and works.

In [1]:
import nltk

nltk.download('gutenberg')
nltk.download('punkt_tab')

from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/iafoslien/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/iafoslien/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# Compute basic linguistic stats for a single book
# Compute basic linguistic stats for a single book
def analyze_book(book, ttr_sample_size=10000):
    words = gutenberg.words(book)
    sents = gutenberg.sents(book)
    vocab = set(w.lower() for w in words if w.isalpha())

    word_count = len(words)
    sentence_count = len(sents)
    vocab_size = len(vocab)

    # lowercase alpha words only, used for the fair vocabulary comparison below
    alpha_words = [w.lower() for w in words if w.isalpha()]
    sample = alpha_words[:ttr_sample_size]
    sample_vocab = set(sample)

    return {
        'book': book,
        'word_count': word_count,
        'sentence_count': sentence_count,
        'vocab_size': vocab_size,
        # raw ratio over the whole book, not fairly comparable across different lengths
        'type_token_ratio_raw': round(vocab_size / word_count, 4),
        # ratio computed on the same size sample from each book, fair comparison
        'type_token_ratio_sample': round(len(sample_vocab) / len(sample), 4),
        'avg_sentence_length': round(word_count / sentence_count, 2)
    }

In [5]:
# Run the analysis on each book and collect the results
books = ['austen-emma.txt', 'melville-moby_dick.txt', 'carroll-alice.txt']
results = [analyze_book(book) for book in books]
results

[{'book': 'austen-emma.txt',
  'word_count': 192427,
  'sentence_count': 7752,
  'vocab_size': 7079,
  'type_token_ratio_raw': 0.0368,
  'type_token_ratio_sample': 0.1762,
  'avg_sentence_length': 24.82},
 {'book': 'melville-moby_dick.txt',
  'word_count': 260819,
  'sentence_count': 10059,
  'vocab_size': 16948,
  'type_token_ratio_raw': 0.065,
  'type_token_ratio_sample': 0.2816,
  'avg_sentence_length': 25.93},
 {'book': 'carroll-alice.txt',
  'word_count': 34110,
  'sentence_count': 1703,
  'vocab_size': 2569,
  'type_token_ratio_raw': 0.0753,
  'type_token_ratio_sample': 0.149,
  'avg_sentence_length': 20.03}]

Vocabulary richness is now computed two ways. The raw type token ratio
uses each book's full word count and isn't fairly comparable across
books of different lengths. The sample based ratio uses the first
10,000 words from each book, giving an apples to apples comparison
of vocabulary variety.

In [6]:
pip install pandas plotly

  Using cached pandas-3.0.5-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.5-cp311-cp311-macosx_11_0_arm64.whl (10.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 7.4 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 7.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]s]
Note: you may need to restart the kernel to use updated packages.


In [7]:
# turn the results list into a table for easier viewing and plotting
import pandas as pd

df = pd.DataFrame(results)
df

,book,word_count,sentence_count,vocab_size,type_token_ratio_raw,type_token_ratio_sample,avg_sentence_length
0,austen-emma.txt,192427,7752,7079,0.0368,0.1762,24.82
1,melville-moby_dick.txt,260819,10059,16948,0.0650,0.2816,25.93
2,carroll-alice.txt,34110,1703,2569,0.0753,0.1490,20.03


### Comparing vocabulary richness fairly

The chart below shows type token ratio for each book using the same
10,000 word sample size, so the three books are compared on equal
footing rather than penalizing the shorter ones.

In [8]:
# compare vocabulary richness across books
import plotly.express as px

fig = px.bar(
    df,
    x='book',
    y='type_token_ratio_sample',
    title='Vocabulary Richness by Book (10,000 word sample)',
    labels={'book': 'Book', 'type_token_ratio_sample': 'Type Token Ratio'},
    color_discrete_sequence=['#2a78d6']
)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

Moby Dick clearly has the richest vocabulary of the three, consistent
with its reputation for dense, varied language. Alice in Wonderland
comes out lowest here, a more sensible result than the raw ratio gave,
since that number was skewed by Alice being the shortest book.